<a href="https://colab.research.google.com/github/safar1-gg/mac237-labs/blob/main/MAC237_LAB1_S02_Cryptographic_Tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MAC 237 Graded Lab 1, Session 2: hashing and message authentication

**Wednesday 16 September 2026. Room B124, or a browser, whichever you are sitting at.**

This notebook is the graded work for session 2. Every cell uses only the Python 3 standard
library, so it runs unchanged in Google Colab in a browser or in Python 3 on the Kali image
in Room B124.

**Nothing in this notebook contacts any host outside it.** Every byte it works on is one it
generated, and there is nothing here to send anywhere.

Three questions, in order. Does a hash notice a change. Does a hash keep a secret. And what
do you have to add to a hash before it can tell you who a message came from. The answers are
yes, no, and a key.

The rest of the hour in B124 is ungraded Kali practice, named in the lab document.

## Section 1. Your personalization token

Put your own student ID in `sid` and run the cell. Use the same student ID you used in
session 1. The token is different every session because the timestamp is part of it, which
is fine: what matters is that it is yours and that it appears in the output you save.

In [1]:
import hashlib, hmac, time, datetime

sid = '24271242'
stamp = datetime.datetime.now().strftime('%Y%m%d-%H%M')
TOKEN = hashlib.sha256((sid + '-' + stamp).encode()).hexdigest()[:16]
print('MAC237 Lab 1 token:', TOKEN)

MAC237 Lab 1 token: 00b0d3cc9be24e83


## Section 2. A hash notices a change

Hash a sentence. Change one letter of it. Hash it again. The interesting number is not that
the two digests differ, which you would expect, but by how much: about half of the 256 bits,
for a change of one letter in forty. That property has a name, the avalanche effect, and it
is what makes a digest usable as a fingerprint. A function that changed a little when the
input changed a little would let an attacker steer it.

In [2]:
MESSAGE  = 'MAC237 lab 1 integrity check for ' + TOKEN
TAMPERED = MESSAGE.replace('integrity', 'integrlty', 1)     # one letter, i to l

d1 = hashlib.sha256(MESSAGE.encode()).digest()
d2 = hashlib.sha256(TAMPERED.encode()).digest()

def bits_differing(a, b):
    return sum(bin(x ^ y).count('1') for x, y in zip(a, b))

flipped = bits_differing(d1, d2)

print('message  :', MESSAGE)
print('tampered :', TAMPERED)
print('digest 1 :', d1.hex())
print('digest 2 :', d2.hex())
print()
print(f'characters changed in the message : 1 of {len(MESSAGE)}')
print(f'bits changed in the digest        : {flipped} of 256')

assert d1 != d2, 'two different messages produced the same digest'
assert 80 <= flipped <= 176, 'a one character change should flip roughly half the bits'
assert hashlib.sha256(MESSAGE.encode()).digest() == d1, 'the same input must hash the same way'
print()
print('  CHECK PASS: one character in, about half the digest bits out, every time')

message  : MAC237 lab 1 integrity check for 00b0d3cc9be24e83
tampered : MAC237 lab 1 integrlty check for 00b0d3cc9be24e83
digest 1 : 65c01a3e0da3dbdf5192305616929c5fe81655ec551b83f478d25427fb2b1cea
digest 2 : 8a3422499ba1b1b9bb0cd54a1cda7a854c79c0abaa7cd7ea9a42b5e47f784223

characters changed in the message : 1 of 49
bits changed in the digest        : 132 of 256

  CHECK PASS: one character in, about half the digest bits out, every time


## Section 3. A hash does not keep a secret

A digest is not encryption and it was never meant to be. It is one way, which stops you
reading the input back out of it, and that is a different thing from stopping an attacker
learning the input. If the set of possible inputs is small, an attacker does not need to
reverse anything. They hash every candidate and compare.

Below, a four digit personal identification number, drawn from your own token so it is yours,
is stored the way far too many systems have stored one: as a bare SHA-256 digest. Then the
attack runs. All ten thousand candidates, in a plain Python loop, on whatever machine you are
sitting at.

In [3]:
PIN = '%04d' % (int(TOKEN[:4], 16) % 10000)
stored_digest = hashlib.sha256(PIN.encode()).hexdigest()

print('token          :', TOKEN)
print('stored digest  :', stored_digest)
print('this digest is the whole of what the attacker steals from the database')
print()

start = time.perf_counter()
recovered, tried = None, 0
for guess in range(10000):
    tried += 1
    candidate = '%04d' % guess
    if hashlib.sha256(candidate.encode()).hexdigest() == stored_digest:
        recovered = candidate
        break
elapsed = time.perf_counter() - start

print(f'recovered      : {recovered}')
print(f'guesses needed : {tried} of 10000')
print(f'time taken     : {elapsed:.4f} seconds')

assert recovered == PIN, 'the search did not recover the value that was hashed'
print()
print('  CHECK PASS: a bare digest of a small secret is not a secret')

token          : 00b0d3cc9be24e83
stored digest  : 36232231716e856d64c4a8a33f09535b743ecf1b98ca8952458a9802f2022301
this digest is the whole of what the attacker steals from the database

recovered      : 0176
guesses needed : 177 of 10000
time taken     : 0.0004 seconds

  CHECK PASS: a bare digest of a small secret is not a secret


Two defenses answer that attack and neither of them is a better hash function. A per-user
random salt means the attacker cannot precompute one table and use it against everybody. A
work factor means each of those ten thousand guesses costs real time instead of a microsecond.
You will measure what a work factor costs an attacker, and what it costs your own login page,
in Lab 3, session 11.

## Section 4. A key turns a fingerprint into a signature of sorts

The digest in Section 2 caught a change, but anyone who can change the message can recompute
the digest, so on its own it proves nothing about who sent it. A message authentication code
fixes that by mixing in a key that only the two parties hold. Now recomputing the tag requires
the key, so a valid tag says two things at once: this message is unmodified, and it came from
somebody holding the key.

`hmac.compare_digest` is used instead of `==` on purpose. An ordinary comparison returns as
soon as it finds a byte that differs, so how long it took leaks how much of the tag the
attacker got right, and an attacker who can measure that can build a valid tag one byte at a
time. `compare_digest` takes the same time either way.

In [4]:
KEY       = hashlib.sha256((TOKEN + '-mac237-lab1-key').encode()).digest()
WRONG_KEY = hashlib.sha256((TOKEN + '-not-the-key').encode()).digest()

def tag(key, message):
    return hmac.new(key, message, hashlib.sha256).digest()

def verify(key, message, candidate):
    return hmac.compare_digest(tag(key, message), candidate)

message  = ('transfer 100 dollars to account 4021, authorized by ' + TOKEN).encode()
tampered = message.replace(b'100', b'900')
t = tag(KEY, message)

print('token     :', TOKEN)
print('message   :', message.decode())
print('tampered  :', tampered.decode())
print('tag       :', t.hex())
print()
print('holder of the key, original message :', verify(KEY, message, t))
print('holder of the key, tampered message :', verify(KEY, tampered, t))
print('wrong key, original message         :', verify(WRONG_KEY, message, t))

assert verify(KEY, message, t), 'the tag should accept the message it was made for'
assert not verify(KEY, tampered, t), 'the tag should reject a modified message'
assert not verify(WRONG_KEY, message, t), 'the wrong key should not verify the tag'
assert tag(WRONG_KEY, message) != t, 'two different keys produced the same tag'
print()
print('  CHECK PASS: the tag accepts this message, under this key, and nothing else')
print('  token:', TOKEN)

token     : 00b0d3cc9be24e83
message   : transfer 100 dollars to account 4021, authorized by 00b0d3cc9be24e83
tampered  : transfer 900 dollars to account 4021, authorized by 00b0d3cc9be24e83
tag       : 91262cbb5fdbf05fc5be3d084dce39a39a7e0d85cb1a8b0926209e59d62bf443

holder of the key, original message : True
holder of the key, tampered message : False
wrong key, original message         : False

  CHECK PASS: the tag accepts this message, under this key, and nothing else
  token: 00b0d3cc9be24e83


## Closing this session

Three `CHECK PASS` lines must appear above.

At minute 45, whichever platform you are on:

1. **Save the notebook into your repository**, at
   `lab1/MAC237_LAB1_S02_Cryptographic_Tools.ipynb`. In Colab: File, then Save a copy in
   GitHub. On Kali: save it into your clone, then `git add -A`, `git commit`, `git push`.
2. Open the commit on github.com and confirm the notebook is there with your token visible in
   the saved output.
3. Screenshots, both showing your own username or Google account name and your token:
   `lab1_s02_avalanche.png`, the two digests and the bit count from Section 2; and
   `lab1_s02_hmac.png`, the three True and False lines from Section 4.
4. Sign out before you leave and capture `lab1_s02_logout.png`.

## What goes in the report

1. Copy the two digests from Section 2 and the bit count. Then say what would be wrong with a
   hash function whose digest changed in only two or three bits when one character of the
   input changed, in terms of what an attacker could then do that they cannot do now.
2. Report how long the Section 3 search took on your machine and what that implies for a
   six digit code, which is a hundred times larger. Then name the two defenses and say which
   one of them would still be needed if every user picked a long random password.
3. Section 4 gives three results. Take the third one, the wrong key with the original message,
   and say what it proves that the second one does not.
4. Name which of these tools you would choose for verifying a downloaded installer, for
   proving to a server that an application programming interface call came from your
   application, and for keeping a message unreadable in transit. One sentence each. One of the
   three is not in this notebook at all, and saying so is the correct answer.
5. Cite CTPE Chapter 2 on cryptographic hash functions and message authentication.